# Day 06 Lab 1: Qwen3.5-4B Text SFT with Unsloth on RunPod

이 노트북은 Day-06의 기본 실습 경로인 `Qwen3.5-4B + bf16 LoRA + Unsloth + RunPod`를 텍스트 데이터 기준으로 끝까지 완주

### 학습 목표

1. bf16 LoRA SFT 파이프라인을 처음부터 끝까지 완주한다
2. LoRA adapter가 base model의 어떤 부분을 바꾸는지 이해한다
3. 학습 전/후 응답 차이를 정성적으로 비교한다
4. train loss가 줄어드는 과정을 시각적으로 확인한다

## 왜 full finetuning이 아니라 LoRA인가

- 목표는 최고 성능 경쟁이 아니라 `install -> train -> save -> infer`를 재현하는 것
- full finetuning은 업데이트해야 할 파라미터와 저장물이 커서 RunPod 실습에서 실패 확률이 높습니다.
- LoRA는 base weight 전체를 다시 저장하지 않고 adapter만 학습하므로 저장 결과가 `adapter/checkpoint` 중심으로 남습니다.
- 이번 실습의 기본 경로는 `bf16 LoRA`입니다. `QLoRA`는 아닙니다.
- 이번 Lab 1은 오직 텍스트 SFT만 다룹니다.


In [ ]:
# NOTE: uv pip 사용. RunPod 환경에서는 uv가 기본 설치되어 있다고 가정합니다.
%%capture
import os
import importlib.util

if importlib.util.find_spec("torch") is None:
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0 datasets accelerate peft pandas openai matplotlib
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(
    "Installation cell finished. If a later import fails, restart the kernel once and continue."
)

### Configuration

이 셀은 실습 중 바꿀 수 있는 값들을 한 곳에 모아 둔 설정 셀입니다.

- 기본 모델은 `Qwen/Qwen3.5-4B`입니다.
- `LOAD_IN_4BIT = False`로 두어 bf16 LoRA 기본 경로를 유지합니다.
- 비교용 프롬프트도 여기서 미리 고정해 두고, base 응답은 학습 전에 저장합니다.


In [ ]:
from pathlib import Path
import random
import pandas as pd
import torch

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_CANDIDATES = [
    "Qwen/Qwen3.5-4B",
    "Qwen/Qwen3.5-2B",  # fallback: OOM 시 사용
    # "Qwen/Qwen3.5-0.8B",  # emergency fallback (모델 확인 후 활성화)
]
MODEL_INDEX = 0  # 0: 4B, 1: 2B
MODEL_NAME = MODEL_CANDIDATES[MODEL_INDEX]

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = False
LOAD_IN_16BIT = True
FULL_FINETUNING = False

# NOTE: 데이터셋 경로 정의
TRAIN_PATH = Path("sft_train.jsonl")
EVAL_PATH = Path("sft_eval.jsonl")
OUTPUT_DIR = Path("outputs/lab1_qwen35_sft")
ADAPTER_DIR = Path("artifacts/lab1_qwen35_sft_adapter")

USE_HF_DATASET_FALLBACK = True
HF_DATASET_NAME = "nlpai-lab/kullm-v2"
HF_DATASET_SPLIT = "train"
HF_MAX_TRAIN_SAMPLES = 3000
HF_MAX_EVAL_SAMPLES = 300

OPENAI_MODEL_NAME = "gpt-5.4-mini"  # NOTE: 사용 전 OpenAI 최신 모델명을 확인하세요
OPENAI_DATA_MODE = "seed_plus_expand"  # seed_only | expand_only | seed_plus_expand
OPENAI_MAX_SOURCE_CHUNKS = 6
OPENAI_EXAMPLES_PER_CHUNK = 2
OPENAI_MAX_PUBLIC_EXPANSIONS = 8
OPENAI_BOOTSTRAP_SAMPLE_SIZE = 400
OPENAI_WRITE_TO_SFT_TRAIN = False
OPENAI_SEED_DRAFT_PATH = Path("sft_train.openai_seed_draft.jsonl")
OPENAI_EXPANSION_DRAFT_PATH = Path("sft_train.openai_public_expansion.jsonl")
OPENAI_FINAL_DRAFT_PATH = Path("sft_train.openai_combined_draft.jsonl")

SYSTEM_PROMPT = """당신은 AgentOps와 Observability를 설명하는 한국어 운영 도우미입니다. 
답변은 정확하고 실행 가능해야 하니 꼭 검증 후 답변하길 바랍니다."""

COMPARISON_PROMPTS = [
    "AgentOps 관점에서 TTFE와 E2E latency의 차이를 한국어로 설명해 줘.",
    "에이전트 장애가 발생했을 때 observability runbook의 첫 5단계를 순서대로 요약해 줘.",
    "SLA breach가 의심될 때 운영자가 가장 먼저 확인해야 할 지표를 3가지 제안해 줘.",
    "prompt optimization playbook이 필요한 상황을 실제 운영 사례처럼 설명해 줘.",
    "error rate가 급증했지만 latency는 정상일 때 가능한 원인 가설을 정리해 줘.",
]

bf16_supported = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
gpu_stats = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
START_GPU_MEMORY = (
    round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    if torch.cuda.is_available()
    else 0.0
)
MAX_GPU_MEMORY = (
    round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3) if gpu_stats else 0.0
)

config_summary = {
    "model_name": MODEL_NAME,
    "fallback_order": MODEL_CANDIDATES,
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "load_in_16bit": LOAD_IN_16BIT,
    "bf16_supported": bf16_supported,
    "train_path": str(TRAIN_PATH),
    "eval_path": str(EVAL_PATH),
    "use_hf_dataset_fallback": USE_HF_DATASET_FALLBACK,
    "hf_dataset_name": HF_DATASET_NAME,
    "openai_model_name": OPENAI_MODEL_NAME,
    "openai_data_mode": OPENAI_DATA_MODE,
    "output_dir": str(OUTPUT_DIR),
    "adapter_dir": str(ADAPTER_DIR),
}

print(
    {"gpu": gpu_name, "max_memory_gb": MAX_GPU_MEMORY, "bf16_supported": bf16_supported}
)
config_summary

### Unsloth

이 셀은 아직 학습되지 않은 Qwen3.5 base 모델을 불러옵니다. 여기서는 4bit가 아니라 16bit(bf16/fp16) LoRA 경로를 사용합니다.

자주 막히는 지점:
- OOM이 나면 가장 먼저 `MODEL_INDEX = 1`로 바꿔 `2B`를 다시 시도하세요.
- Hugging Face 인증이 안 되어 있으면 모델 다운로드 단계에서 멈출 수 있습니다.

`tokenizer.pad_token = tokenizer.eos_token`은 Qwen 토크나이저에 별도 pad_token이 없을 때의 fallback입니다. SFTTrainer가 padding 위치의 loss를 자동으로 마스킹(`labels = -100`)하므로 이번 실습에서는 안전합니다. 다만, 직접 training loop를 구현할 때는 padding 토큰의 loss를 반드시 `-100`으로 마스킹해야 합니다.

In [ ]:
from unsloth import FastLanguageModel

# OOM 시 자동으로 더 작은 모델로 fallback
_loaded = False
for _idx in range(MODEL_INDEX, len(MODEL_CANDIDATES)):
    try:
        MODEL_NAME = MODEL_CANDIDATES[_idx]
        print(f"Loading {MODEL_NAME}...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=LOAD_IN_4BIT,
            load_in_16bit=LOAD_IN_16BIT,
            full_finetuning=FULL_FINETUNING,
        )
        _loaded = True
        break
    except (RuntimeError,) as e:
        if "out of memory" in str(e).lower() and _idx < len(MODEL_CANDIDATES) - 1:
            print(f"⚠ OOM with {MODEL_NAME}. Falling back to {MODEL_CANDIDATES[_idx + 1]}...")
            import torch
            torch.cuda.empty_cache()
        else:
            raise

if not _loaded:
    raise RuntimeError("All model candidates failed to load.")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(
    {
        "loaded_model": MODEL_NAME,
        "pad_token": tokenizer.pad_token,
        "eos_token": tokenizer.eos_token,
    }
)

이제 LoRA adapter를 붙입니다.

- `r=16` (rank): adapter의 차원 수. 클수록 표현력이 높지만 VRAM을 더 쓴다. r=16은 교육용 기본값으로, 표현력과 메모리 사이의 균형점이다.
- `lora_alpha=16`: adapter 업데이트의 스케일 계수. alpha/r = 1.0이면 기본 스케일. alpha를 높이면 adapter의 영향이 커진다.
- `target_modules`: 아래 7개 모듈 전체에 LoRA를 적용한다.
  - **Attention (q/k/v/o)**: "무엇에 주의를 기울이는가"를 조정. 도메인 문맥 이해에 직접 영향.
  - **FFN (gate/up/down)**: "각 토큰의 표현을 어떻게 변환하는가"를 조정. 도메인 지식 주입에 기여.
  - Attention만 적용(4개)하면 VRAM ~30% 절감되지만 학습 능력이 제한된다. 전체 적용(7개)이 도메인 적응에 더 효과적이다.
- `use_gradient_checkpointing="unsloth"`: 중간 activation을 재계산하여 VRAM ~30-40% 절감. `"unsloth"` 모드는 표준 PyTorch checkpointing보다 overhead가 적다.
- `use_rslora=False`: 표준 $\alpha/r$ 스케일링을 사용한다. `True`로 바꾸면 rank-stabilized LoRA($\alpha/\sqrt{r}$)가 적용되어 rank 변경 시 학습 안정성이 유지된다.
- full finetuning이 아니라 adapter만 학습한다는 점을 코드로 확인하는 단계입니다.

In [ ]:
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=TARGET_MODULES,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    max_seq_length=MAX_SEQ_LENGTH,
)

model.print_trainable_parameters()


<a name="Data"></a>
### Data Prep

SFT 학습 데이터는 모델이 "어떤 질문에 어떻게 답해야 하는가"를 가르치는 예시 모음입니다.

#### SFT 데이터의 기본 구조

각 학습 샘플은 세 개의 필드로 구성됩니다:

| 필드 | 역할 | 예시 |
|------|------|------|
| `instruction` | 모델에게 시킬 작업 지시 | "TTFE와 E2E latency의 차이를 설명해 줘" |
| `input` | 추가 맥락 (비어 있을 수 있음) | "에이전트가 3개의 tool을 순차 호출하는 시나리오에서" |
| `output` | 모델이 생성해야 할 정답 | "TTFE는 첫 번째 토큰이 나올 때까지의 시간이고..." |

이 세 필드가 하나의 **chat template 문자열**로 합쳐져서 모델에 입력됩니다:
```
<|im_start|>system\n시스템 프롬프트<|im_end|>
<|im_start|>user\ninstruction + input<|im_end|>
<|im_start|>assistant\noutput<|im_end|>
```

#### 좋은 SFT 데이터의 조건

- **품질 > 양**: 1000건의 부정확한 데이터보다 200건의 정확한 데이터가 낫습니다.
- **Groundedness**: 답변이 실제 문서나 사실에 근거해야 합니다. 모델이 없는 내용을 지어내면 안 됩니다.
- **다양성**: 같은 유형의 질문만 반복되면 모델이 한 가지 패턴만 학습합니다. metric 설명, runbook 요약, incident 정리 등 다양한 task type을 섞어야 합니다.
- **일관성**: system prompt와 output의 어조/형식이 일관되어야 합니다.

#### 데이터 흐름

```
seed 문서 → synthetic generation → 품질 검수 → sft_train.jsonl → chat template 포맷 → 모델 학습
```

Lab 0에서 이미 `sft_train.jsonl`을 만들었다면 그것을 그대로 사용합니다. 없다면 아래에서 HF 공개 데이터셋을 fallback으로 사용합니다.

#### Hugging Face 검토 결과
- `nlpai-lab/kullm-v2`: 한국어, `instruction/input/output` 스키마, train 153k rows, `apache-2.0`, 실습 연결이 가장 쉬움
- `heegyu/open-korean-instructions`: 한국어 데이터 규모는 크지만 `<usr>/<bot>` 형식이라 추가 정제가 더 필요함
- `thunder-research-group/SNU_Thunder-synthetic-instruction-following`: 한국어+영어 혼합이며 연구/비상업 중심 안내가 있어 기본 실습 데이터로는 우선순위를 낮춤

따라서 기본 한국어 fallback은 `nlpai-lab/kullm-v2`로 고정합니다. 다만 Day-06의 핵심 도메인은 AgentOps / Observability이므로, **수업용 최우선 데이터는 여전히 Lab 0의 도메인 데이터**입니다.

In [ ]:
from datasets import load_dataset

DATA_SOURCE = None

if TRAIN_PATH.exists():
    DATA_SOURCE = "local_lab0"
    raw_train_dataset = load_dataset("json", data_files=str(TRAIN_PATH), split="train")
    if EVAL_PATH.exists():
        raw_eval_dataset = load_dataset(
            "json", data_files=str(EVAL_PATH), split="train"
        )
    else:
        eval_size = (
            1
            if len(raw_train_dataset) < 10
            else max(1, int(len(raw_train_dataset) * 0.1))
        )
        split_dataset = raw_train_dataset.train_test_split(
            test_size=eval_size, seed=SEED
        )
        raw_train_dataset = split_dataset["train"]
        raw_eval_dataset = split_dataset["test"]
        print(
            f"{EVAL_PATH} not found. Built a small hold-out eval split from local train data."
        )
elif USE_HF_DATASET_FALLBACK:
    DATA_SOURCE = "hf_fallback"
    raw_dataset = load_dataset(HF_DATASET_NAME, split=HF_DATASET_SPLIT)
    required_columns = {"instruction", "input", "output"}
    missing_columns = required_columns - set(raw_dataset.column_names)
    if missing_columns:
        raise ValueError(f"HF dataset is missing required columns: {missing_columns}")

    target_total = HF_MAX_TRAIN_SAMPLES + HF_MAX_EVAL_SAMPLES
    if len(raw_dataset) > target_total:
        raw_dataset = raw_dataset.shuffle(seed=SEED).select(range(target_total))
    else:
        raw_dataset = raw_dataset.shuffle(seed=SEED)

    split_dataset = raw_dataset.train_test_split(
        test_size=min(HF_MAX_EVAL_SAMPLES, max(1, int(len(raw_dataset) * 0.1))),
        seed=SEED,
    )
    raw_train_dataset = split_dataset["train"]
    raw_eval_dataset = split_dataset["test"]
else:
    raise FileNotFoundError(
        "No local Lab 0 dataset was found and HF fallback is disabled. Provide sft_train.jsonl or enable HF fallback."
    )

# 빈 데이터셋 가드
if len(raw_train_dataset) == 0:
    raise ValueError(f"학습 데이터가 0건입니다. 데이터 파일을 확인하세요.")

# HF fallback 도메인 불일치 경고
if DATA_SOURCE == "hf_fallback":
    print("=" * 60)
    print("⚠ WARNING: HF fallback 데이터는 일반 한국어 instruction입니다.")
    print("  AgentOps 도메인 비교 프롬프트에서 큰 차이가 나지 않을 수 있습니다.")
    print("  도메인 성능을 보려면 Lab 0의 도메인 데이터를 사용하세요.")
    print("=" * 60)

print(
    {
        "data_source": DATA_SOURCE,
        "train_rows": len(raw_train_dataset),
        "eval_rows": len(raw_eval_dataset),
        "columns": raw_train_dataset.column_names,
    }
)

먼저 원본 데이터 예시를 확인합니다. 수강생이 여기서 `instruction`, `input`, `output`이 어떤 의미인지 눈으로 확인해야 다음 포맷팅 셀을 이해하기 쉽습니다.

특히 HF fallback을 쓰는 경우에는 아래 preview로 정말 한글 instruction 데이터가 들어왔는지 먼저 확인하세요. 한국어가 충분하지 않거나 형식이 깨져 있으면 이 단계에서 바로 드러납니다.


In [ ]:
preview_columns = [
    column
    for column in ["instruction", "input", "output", "task_type", "source_doc"]
    if column in raw_train_dataset.column_names
]
display(
    raw_train_dataset.select(range(min(3, len(raw_train_dataset)))).to_pandas()[
        preview_columns
    ]
)


In [ ]:
# ---------- train/eval 오염 검사 ----------
train_instructions = set(raw_train_dataset["instruction"])
eval_instructions = set(raw_eval_dataset["instruction"])
overlap = train_instructions & eval_instructions
if overlap:
    print(f"⚠ WARNING: {len(overlap)} instructions appear in both train and eval sets:")
    for inst in list(overlap)[:3]:
        print(f"  - {inst[:80]}")
else:
    print(f"✓ No instruction overlap between train ({len(train_instructions)}) and eval ({len(eval_instructions)}) sets")

이제 raw row를 실제 학습 문자열로 바꿉니다. 초심자가 가장 많이 막히는 지점이 “JSONL 한 줄이 실제로 모델에 어떤 형태로 들어가는가”이므로, 변환 함수와 결과 예시를 둘 다 남깁니다.


In [ ]:
def build_user_message(example):
    instruction = example["instruction"].strip()
    extra_input = str(example.get("input", "") or "").strip()
    if extra_input:
        return f"{instruction}\n\n추가 입력:\n{extra_input}"
    return instruction


def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(example)},
        {"role": "assistant", "content": example["output"].strip()},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }


train_dataset = raw_train_dataset.map(
    format_example, desc="Formatting SFT train dataset"
)
eval_dataset = raw_eval_dataset.map(format_example, desc="Formatting SFT eval dataset")

print(train_dataset[0]["text"][:700])


In [ ]:
print("=" * 60)
print("[ Chat Template 변환 결과 전체 보기 ]")
print("=" * 60)
print(train_dataset[0]["text"])
print("=" * 60)
print(f"Total characters: {len(train_dataset[0]['text'])}")

In [ ]:
sample_size = min(500, len(train_dataset))
sample_dataset = train_dataset.select(range(sample_size))
lengths = [len(tokenizer.encode(ex["text"])) for ex in sample_dataset]
length_series = pd.Series(lengths)

print(f"=== Token Length Distribution (sampled {sample_size} rows) ===")
display(length_series.describe())
print(
    f"\nMAX_SEQ_LENGTH({MAX_SEQ_LENGTH}) 초과 비율: {(length_series > MAX_SEQ_LENGTH).mean():.1%}"
)

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(lengths, bins=30, edgecolor="black", alpha=0.7)
    ax.axvline(
        MAX_SEQ_LENGTH,
        color="red",
        linestyle="--",
        label=f"MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}",
    )
    ax.set_xlabel("Token count")
    ax.set_ylabel("Frequency")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib 미설치. 위 통계표를 참고하세요.")

---
## 데이터 준비 안내

### Lab 0에서 데이터를 준비한 경우
`lab0_synthetic_data.ipynb`에서 생성한 `sft_train.jsonl`과 `sft_eval.jsonl`을 이 노트북과 같은 디렉토리에 두세요.

### Lab 0을 건너뛴 경우
아래 두 가지 fallback 경로 중 하나가 자동으로 적용됩니다:
1. **폴백 데이터셋**: `data/sft_train_fallback.jsonl`이 있으면 이를 사용
2. **HuggingFace 공개 데이터셋**: `nlpai-lab/kullm-v2` (일반 한국어 instruction)를 자동 다운로드

> **참고:** HuggingFace fallback은 일반 한국어 데이터이므로, AgentOps 도메인 비교 프롬프트에서 큰 차이가 나지 않을 수 있습니다. 도메인 성능을 보려면 Lab 0의 도메인 데이터를 사용하세요.

### Before training smoke test

- 목적 1: chat template와 토크나이저가 정상인지 빠르게 확인
- 목적 2: 학습 후 비교할 base 응답을 미리 저장
- 목적 3: 긴 학습 전에 프롬프트 자체가 적절한지 점검

In [ ]:
import re

FastLanguageModel.for_inference(model)


def make_generation_inputs(current_tokenizer, prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    try:
        return current_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            enable_thinking=False,  # Qwen3.5 thinking mode 비활성화
        )
    except TypeError:
        # Unsloth 토크나이저가 enable_thinking을 지원하지 않는 경우
        return current_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )


def generate_response(current_model, current_tokenizer, prompt, max_new_tokens=220):
    input_ids = make_generation_inputs(current_tokenizer, prompt).to(
        current_model.device
    )
    attention_mask = torch.ones_like(input_ids)
    with torch.inference_mode():
        outputs = current_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # 다양한 응답을 보고 싶으면 아래로 바꿔 보세요:
            # do_sample=True, temperature=0.6, top_p=0.95,
            use_cache=True,
        )
    generated_tokens = outputs[0][input_ids.shape[-1] :]
    text = current_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    # Qwen3.5 thinking 태그가 남아 있으면 제거
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return text


smoke_test_prompt = COMPARISON_PROMPTS[0]
base_smoke_test_response = generate_response(model, tokenizer, smoke_test_prompt)
base_comparison_responses = [
    generate_response(model, tokenizer, prompt) for prompt in COMPARISON_PROMPTS
]

pd.DataFrame(
    [{"prompt": smoke_test_prompt, "base_response_preview": base_smoke_test_response}]
)

<a name="Train"></a>
### Train the model

이제 학습을 실행합니다. 공식 Unsloth notebook처럼 먼저 학습 모드로 전환하고, 보수적인 `max_steps`로 빠르게 완주할 수 있게 둡니다.

**GPU 자동 감지**: 아래 코드는 GPU 메모리를 감지하여 하이퍼파라미터를 자동 조정합니다.

| GPU 환경 | batch_size | grad_accum | max_steps | effective batch |
|---------|-----------|------------|-----------|----------------|
| **H100 80GB / A100 80GB** | 8 | 2 | 120 | 16 |
| 24GB (RTX 4090, L4 등) | 2 | 4 | 60 | 8 |
| 소형 GPU | 1 | 8 | 30 | 8 |

- `per_device_train_batch_size` × `gradient_accumulation_steps` = **effective batch size**. 작은 GPU에서 큰 effective batch를 흉내내는 기법이다.
- `warmup_steps=5`: 학습 초기 5 step 동안 learning rate를 0에서 목표값까지 서서히 올린다.
- `learning_rate=2e-4`: QLoRA 논문(Dettmers et al., 2023)과 Unsloth 공식 기본값. 일반적인 LoRA lr 범위는 1e-5 ~ 5e-4.
- `optim="adamw_8bit"`: 8-bit optimizer로 ~2GB VRAM 절약. fp32 AdamW 대비 품질 손실은 미미.
- `packing=False`는 데이터가 어떤 문자열로 들어가는지 추적하기 쉬워서 교육용에 유리합니다.

**max_steps가 충분한가?** (공식: `steps_per_epoch = len(train_dataset) / (batch_size × grad_accum_steps)`)

| 데이터셋 크기 | Steps/Epoch (H100) | Steps/Epoch (24GB) | 커버리지 (H100 120steps) | 커버리지 (24GB 60steps) |
|---|---|---|---|---|
| 150건 | ~9 | ~19 | ~13 epochs | ~3.2 epochs |
| 250건 | ~16 | ~31 | ~7.5 epochs | ~1.9 epochs |
| 3000건 (HF fallback) | ~188 | ~375 | 0.6 epoch | 0.16 epoch |

참고: [HuggingFace PEFT LoRA](https://huggingface.co/docs/peft/developer_guides/lora) | [QLoRA Paper](https://arxiv.org/abs/2305.14314)

In [ ]:
from trl import SFTConfig, SFTTrainer

FastLanguageModel.for_training(model)

# ---------- H100/A100 자동 감지: GPU 메모리에 따라 하이퍼파라미터 조정 ----------
if MAX_GPU_MEMORY > 40:
    # H100 80GB / A100 80GB: 대폭 확대 가능
    _batch_size = 8
    _grad_accum = 2
    _max_steps = 120
    print(f"🔧 고용량 GPU 감지 ({gpu_name}, {MAX_GPU_MEMORY}GB)")
    print(f"   → batch_size={_batch_size}, grad_accum={_grad_accum}, max_steps={_max_steps}")
    print(f"   → effective batch = {_batch_size * _grad_accum}")
elif MAX_GPU_MEMORY > 20:
    # 24GB GPU (RTX 4090, L4, A5000 등): 원래 설정
    _batch_size = 2
    _grad_accum = 4
    _max_steps = 60
    print(f"🔧 24GB급 GPU 감지 ({gpu_name}, {MAX_GPU_MEMORY}GB)")
    print(f"   → batch_size={_batch_size}, grad_accum={_grad_accum}, max_steps={_max_steps}")
else:
    # 그 외 소형 GPU
    _batch_size = 1
    _grad_accum = 8
    _max_steps = 30
    print(f"⚠ 소형 GPU ({gpu_name}, {MAX_GPU_MEMORY}GB) — 보수적 설정 적용")

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=_batch_size,
    per_device_eval_batch_size=min(_batch_size, 4),
    gradient_accumulation_steps=_grad_accum,
    warmup_steps=5,
    max_steps=_max_steps,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=20,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=SEED,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    report_to="none",
    packing=False,
    dataset_num_proc=1,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

print(f"GPU = {gpu_name}. Max memory = {MAX_GPU_MEMORY} GB.")
print(f"{START_GPU_MEMORY} GB of memory reserved before training.")


In [ ]:
trainer_stats = trainer.train()
pd.DataFrame([trainer_stats.metrics])


In [ ]:
USED_MEMORY = (
    round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    if torch.cuda.is_available()
    else 0.0
)
USED_MEMORY_FOR_LORA = round(USED_MEMORY - START_GPU_MEMORY, 3)
USED_PERCENTAGE = (
    round((USED_MEMORY / MAX_GPU_MEMORY) * 100, 3) if MAX_GPU_MEMORY else 0.0
)
LORA_PERCENTAGE = (
    round((USED_MEMORY_FOR_LORA / MAX_GPU_MEMORY) * 100, 3) if MAX_GPU_MEMORY else 0.0
)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {USED_MEMORY} GB.")
print(f"Peak reserved memory for training = {USED_MEMORY_FOR_LORA} GB.")
print(f"Peak reserved memory % of max memory = {USED_PERCENTAGE} %.")
print(f"Peak reserved memory for training % of max memory = {LORA_PERCENTAGE} %.")


### 학습 경과 확인

학습이 진행되면 loss가 줄어들어야 합니다.
- loss가 줄지 않으면: learning rate가 너무 크거나, 데이터에 문제가 있을 수 있습니다.
- eval loss가 train loss보다 많이 높아지면: overfitting 신호입니다.
- 60 step은 매우 짧으므로 큰 변화가 없을 수도 있습니다. 정상입니다.

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_logs = [log for log in log_history if "loss" in log]
eval_logs = [log for log in log_history if "eval_loss" in log]

if train_logs:
    train_df = pd.DataFrame(train_logs)
    print("=== Train Loss Progress ===")
    display(train_df[["step", "loss"]])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(train_df["step"], train_df["loss"], marker="o", label="train loss")
    if eval_logs:
        eval_df = pd.DataFrame(eval_logs)
        ax.plot(eval_df["step"], eval_df["eval_loss"], marker="s", label="eval loss")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("Training Progress")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No training logs found.")

In [ ]:
# ---------- Eval Perplexity 계산 ----------
# Perplexity = e^(cross-entropy loss). 낮을수록 모델이 정답 토큰을 더 잘 예측한다는 뜻.
# 단, perplexity가 낮아도 생성 품질이 반드시 좋은 것은 아니다 (overfitting 가능).
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print(f"Eval loss: {eval_results['eval_loss']:.4f}")
print(f"Eval perplexity: {perplexity:.2f}")
print(f"\n해석: perplexity {perplexity:.0f} = 모델이 각 위치에서 평균 {perplexity:.0f}개 후보 중 하나를 고르는 수준의 불확실성")

<a name="Save"></a>
### Saving, loading finetuned models

Day-06에서는 저장 결과가 `adapter/checkpoint` 중심으로 남아야 합니다. 따라서 merge된 full model이 아니라 재사용 가능한 LoRA adapter를 우선 저장합니다.

자주 막히는 지점:
- pod를 재시작하면 ephemeral storage가 날아갈 수 있으므로, 영속 스토리지 정책을 먼저 확인하세요.
- `OUTPUT_DIR` 안의 checkpoint와 `ADAPTER_DIR` 안의 adapter는 역할이 다릅니다.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()

saved_items = {
    "adapter_dir": str(ADAPTER_DIR.resolve()),
    "checkpoint_dir": str(OUTPUT_DIR.resolve()),
    "adapter_files": sorted(path.name for path in ADAPTER_DIR.iterdir()),
}
saved_items


<a name="Inference"></a>
### Base vs tuned inference

이 셀은 같은 프롬프트에서 base와 tuned 응답을 나란히 보여 줍니다.

중요한 점:
- base 응답은 이미 학습 전에 저장해 두었으므로, 여기서는 추가로 base 모델을 다시 올리지 않습니다.
- 이 방식은 `4B` 환경에서 base와 tuned 모델을 동시에 GPU에 올리지 않아도 되어 더 안전합니다.
- 정성 비교에서는 답변 길이보다 도메인 용어 정확도, 단계 설명의 구체성, runbook 적합성을 먼저 보세요.


In [ ]:
FastLanguageModel.for_inference(model)
tuned_comparison_responses = [
    generate_response(model, tokenizer, prompt) for prompt in COMPARISON_PROMPTS
]

comparison_df = pd.DataFrame(
    {
        "prompt": COMPARISON_PROMPTS,
        "base_response": base_comparison_responses,
        "tuned_response": tuned_comparison_responses,
    }
)
comparison_df


### 자가진단 체크포인트

아래 질문에 답할 수 있는지 확인하세요. 답이 떠오르지 않으면 위 이론 셀을 다시 읽어 보세요.

1. **r=16의 의미는?** → LoRA가 업데이트를 표현하는 차원 수. 16차원 공간에서만 가중치 변화를 표현한다. 높이면 표현력↑ VRAM↑, 낮추면 반대.

2. **bf16 LoRA와 QLoRA의 핵심 차이는?** → base weight 정밀도. bf16은 16-bit 그대로, QLoRA는 4-bit로 양자화. 둘 다 adapter만 학습하는 것은 동일.

3. **train loss가 줄었는데 응답이 나아지지 않았다면?** → (a) 데이터 품질 문제: 부정확한 output으로 학습, (b) overfitting: eval loss가 올라가는지 확인, (c) max_steps 부족: 데이터 크기 대비 학습량 확인.

4. **왜 SFT loss는 assistant 응답에만 계산하는가?** → system/user 부분은 "입력"이지 "모델이 생성해야 할 것"이 아니므로. 모델은 "주어진 맥락에서 어떻게 답하는가"만 학습해야 한다.

5. **perplexity 20은 좋은 것인가?** → 모델이 각 토큰 위치에서 20개 후보 중 하나를 고르는 수준. 도메인 데이터에서 base 모델 대비 낮아졌다면 개선된 것. 절대값보다 상대 변화가 중요하다.

## 정리 및 다음 단계

이 노트북에서 완료한 것:
1. Qwen3.5-4B를 bf16 LoRA로 로드하고 adapter를 설정했다
2. 한국어 instruction 데이터를 chat template로 포맷팅했다
3. SFTTrainer로 학습을 실행하고 loss 변화를 확인했다
4. Adapter를 저장하고 base vs tuned 응답을 비교했다

핵심 질문 (스스로 답해 보세요):
- 왜 full finetuning이 아니라 bf16 LoRA를 선택했는가?
- r=16이 의미하는 것은 무엇이고, 더 크거나 작으면 어떻게 되는가?
- train loss가 줄어들었는데 응답 품질이 나아지지 않았다면 원인은 무엇일까?

배포를 위한 다음 단계 (이번 실습 범위 밖):
- `model.merge_and_unload()`: LoRA adapter를 base model에 병합하여 단일 모델로 만듦
- GGUF 변환: `model.save_pretrained_gguf()`로 Ollama/llama.cpp 호환 형식 변환
- **→ Day-07에서 이 adapter를 vLLM으로 서빙하는 방법을 배웁니다.**

다음 단계:
- Lab 2에서는 같은 도메인 데이터로 embedding model을 튜닝한다
- SFT는 응답 생성 품질을, embedding tuning은 검색 품질을 개선한다

> **참고:** Lab 2를 같은 환경에서 이어서 실행하려면 커널을 재시작하세요. transformers 버전이 다릅니다.

참고: [Unsloth Saving Guide](https://unsloth.ai/docs/basics/saving)

### Fallback logic

1. `Qwen3.5-4B` 로딩에서 OOM 또는 다운로드 실패가 나면 `MODEL_INDEX = 1`로 바꾸고 위 셀들부터 다시 실행합니다.
2. `2B`에서도 OOM이 나면 `MODEL_INDEX = 2`로 바꾸고 `0.8B`로 내립니다.
3. 설치 오류면 커널 재시작 후 installation 셀만 다시 실행합니다.
4. 로컬 Lab 0 데이터가 없으면 `USE_HF_DATASET_FALLBACK = True` 상태에서 `nlpai-lab/kullm-v2`를 자동 fallback으로 사용합니다.
5. HF dataset preview에서 한글 비중이나 스키마가 기대와 다르면 다른 공개 데이터셋으로 바로 넘어가지 말고, 먼저 preview 단계에서 형식을 다시 검토합니다.
6. 데이터셋 오류면 `instruction / input / output` 컬럼명을 먼저 맞춥니다.
7. 저장 오류면 RunPod 영속 볼륨 마운트 위치로 `OUTPUT_DIR`, `ADAPTER_DIR`를 바꿉니다.

이 순서를 지키는 이유는 Day-06 기본 설계가 `4B -> 2B -> 0.8B` 완주성 최적화이기 때문입니다. 처음부터 QLoRA나 9B 이상 모델로 우회하지 않습니다.


## Optional exercise

- 위 `COMPARISON_PROMPTS`에 자신이 만든 AgentOps 프롬프트를 추가해 보세요.
- `max_steps`, `MAX_SEQ_LENGTH`, `SYSTEM_PROMPT`를 바꿨을 때 어떤 trade-off가 생기는지 기록해 보세요.
- tuned 응답이 단순히 길어졌는지, 아니면 실제로 더 domain-specific해졌는지 구분해서 메모해 보세요.


In [ ]:
my_prompt = "운영자가 incident review를 할 때 observability 지표를 어떤 순서로 읽어야 하는지 설명해 줘."

# 예시:
# my_base = base_comparison_responses[0]
# my_tuned = generate_response(model, tokenizer, my_prompt)
# print(my_tuned)
